# FX Short-Signal Backtest Notebook

This notebook:

1. Takes a user configuration: `currency`, `start_date`, `end_date`
2. Pulls Bloomberg `PX_LAST` (close, for signal) and `PX_LOW` (daily low, for exit checks)
3. Builds a daily signal using px_low vs 49D SMA(close)
4. Backtests a short strategy
5. Records entries and exits (`TP` or `SL`)
6. Reports summary statistics including PnL and holding periods

## Assumptions used here

- **Entry signal**: `+1` when `px_low < 49D SMA(close)`, else `-1`
- **Trade direction**: short on days with signal `+1`; entry price = that day's `px_low`
- **Take profit (TP)**: exit when `px_low < 199D SMA(close)`
- **Stop loss (SL)**: exit when `px_low > 49D SMA(close)`
- **Entry & exit price**: both recorded as `px_low` on the respective day
- **One position at a time**: a new entry is only opened when there is no existing open position
- **Execution convention**: on each day, the notebook first checks exits for already-open positions, then checks whether to open a new position using that day's close

You can change any of those rules in the backtest section.


In [ ]:
# Optional: install xbbg if needed
# !pip install xbbg --quiet


In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List, Optional

try:
    from xbbg import blp
except ImportError as exc:
    raise ImportError(
        "xbbg is not installed. Install it with `pip install xbbg` and make sure Bloomberg Terminal/API is available."
    ) from exc


## 1) User configuration

Use a Bloomberg ticker such as:

- `USDCAD Curncy`
- `EURUSD Curncy`
- `USDJPY Curncy`

Set `ALLOW_MULTIPLE_POSITIONS = True` to revert to the original behaviour
of opening a new short on every `+1` day regardless of existing positions.


In [ ]:
config = {
    "currency": "USDCAD Curncy",
    "start_date": "2020-01-01",
    "end_date": "2026-04-01",
}

# FIX (improvement): set to True to allow simultaneous overlapping shorts
ALLOW_MULTIPLE_POSITIONS: bool = False

config

## 2) Pull Bloomberg PX_LOW

In [ ]:
def get_spot_data(currency: str, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Pull daily PX_LAST (close) and PX_LOW from Bloomberg using xbbg.

    - PX_LAST is used to build the SMA signal.
    - PX_LOW  is used to evaluate TP / SL exit conditions each day.

    Fetching both fields in a single bdh() call is more efficient and
    guarantees identical date alignment between the two series.
    """
    raw = blp.bdh(
        tickers=currency,
        flds=["px_last", "px_low"],
        start_date=start_date,
        end_date=end_date,
    )

    if raw.empty:
        raise ValueError(
            "No data returned from Bloomberg. Check ticker, dates, and Bloomberg connection."
        )

    # xbbg returns a MultiIndex column (ticker, field) when multiple fields
    # are requested.  Flatten to simple column names.
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = [field for (_, field) in raw.columns]

    raw = raw.rename(columns={"px_last": "spot", "px_low": "low"})
    raw.index = pd.to_datetime(raw.index)
    raw = raw.sort_index().dropna()
    return raw


df = get_spot_data(
    currency=config["currency"],
    start_date=config["start_date"],
    end_date=config["end_date"],
)

print(f"Loaded {len(df)} rows  |  {df.index[0].date()} → {df.index[-1].date()}")
df.head()

## 3) Build indicators and signal

SMAs are now computed on `close` (PX_LAST). The signal fires `+1` when
`px_low < SMA49(close)`. Live spot crossing below “live 50D SMA” is mathematically equivalent to crossing below yesterday-based 49D SMA.


**Intraday 50D SMA vs 49D Close-Based SMA Equivalence**

We start from the Bloomberg-style intraday definition of the 50-day moving average:

$$
SMA_{50}^{intraday}(t)
= \frac{1}{50} \left( \sum_{i=1}^{49} P_{t-i}^{close} + P_t^{live} \right)
$$

A short signal is triggered when live price crosses below this moving average:

$$
P_t^{live} < SMA_{50}^{intraday}(t)

\; \Rightarrow \;


\; \Rightarrow \;

\boxed{
P_t^{live} < SMA_{49}(t)
}
$$


The intraday condition:
  - *live spot vs live 50D SMA*

is exactly equivalent to:
  - *live spot vs 49-day SMA based on past closes only*


In [ ]:
def prepare_signals(data: pd.DataFrame) -> pd.DataFrame:
    """
    Build SMA indicators and the entry signal, all based on `close` (PX_LAST).
    """
    out = data.copy()

    # Shift forward by 1 day to eliminate look-ahead bias
    out["sma_50"]  = out["spot"].rolling(window=49,  min_periods=49).mean().shift(1)
    out["sma_200"] = out["spot"].rolling(window=199, min_periods=199).mean().shift(1)
    
    # Raw signal: +1 when px_low < SMA50(spot), -1 otherwise
    raw_signal = np.where(out["spot"] < out["sma_50"], 1, -1)
    out["signal"] = pd.Series(raw_signal, index=out.index)

    # Row 0 NaN after shift → no-action
    out["signal"] = out["signal"].fillna(-1)

    # Before SMA50 has enough history, force no-action
    out.loc[out["sma_50"].isna(), "signal"] = -1

    return out


df = prepare_signals(df)
df.tail()

## 4) Backtest logic

Each day:

1. Check whether any already-open short positions should be closed
2. If today's signal is `+1` **and** no position is already open, open a new short

A position is closed as:

- `TP` if `px_low < 200D SMA(close)`
- else `SL` if `px_low > 50D SMA(close)`

Entry price is also `px_low` on the entry day.
Realized PnL = `entry_low - exit_low` (short profits when price falls).
Holding period is in **calendar days**.


In [ ]:
def backtest_short_signal(
    data: pd.DataFrame,
    allow_multiple_positions: bool = False,
) -> pd.DataFrame:
    """
    Run the short-signal backtest.

    Parameters
    ----------
    data : pd.DataFrame
        Output of prepare_signals().
    allow_multiple_positions : bool
        If False (default), a new position is only opened when no short
        is currently open.  Set True to replicate the original behaviour
        of opening a new short on every +1 signal day.
    """
    open_positions: List[Dict] = []
    closed_positions: List[Dict] = []
    trade_id = 0

    for date, row in data.iterrows():
        low     = row["low"]     # PX_LOW — used for entry and exit
        sma_50  = row["sma_50"]
        sma_200 = row["sma_200"]
        signal  = row["signal"]

        # ------------------------------------------------------------------
        # 1) Exit logic — check whether today's low touched an exit level
        # ------------------------------------------------------------------
        still_open: List[Dict] = []
        for pos in open_positions:
            exit_reason: Optional[str] = None

            if pd.notna(sma_200) and low < sma_200:
                exit_reason = "TP"
            elif pd.notna(sma_50) and low > sma_50:
                exit_reason = "SL"

            if exit_reason is None:
                still_open.append(pos)
            else:
                holding_days = (date - pos["entry_date"]).days
                pnl = pos["entry_spot"] - low   # short: profit = entry - exit
                closed_positions.append(
                    {
                        "trade_id":     pos["trade_id"],
                        "entry_date":   pos["entry_date"],
                        "entry_spot":   pos["entry_spot"],
                        "exit_date":    date,
                        "exit_spot":    low,
                        "result":       exit_reason,
                        "pnl":          round(pnl, 6),
                        "holding_days": holding_days,
                    }
                )

        open_positions = still_open

        # ------------------------------------------------------------------
        # 2) Entry logic
        # FIX (improvement): only enter if no existing open position
        #   (unless ALLOW_MULTIPLE_POSITIONS is True)
        # ------------------------------------------------------------------
        can_enter = allow_multiple_positions or len(open_positions) == 0
        if signal == 1 and can_enter:
            trade_id += 1
            open_positions.append(
                {
                    "trade_id":   trade_id,
                    "entry_date": date,
                    "entry_spot": low,   # entry price = px_low
                }
            )

    # -----------------------------------------------------------------------
    # Assemble results DataFrame
    # -----------------------------------------------------------------------
    trades = pd.DataFrame(closed_positions) if closed_positions else pd.DataFrame()

    # Append any still-open positions for visibility
    if open_positions:
        open_df = pd.DataFrame(open_positions)
        open_df["exit_date"]    = pd.NaT
        open_df["exit_spot"]    = np.nan
        open_df["result"]       = "OPEN"
        open_df["pnl"]          = np.nan
        open_df["holding_days"] = np.nan
        trades = pd.concat([trades, open_df], ignore_index=True, sort=False)

    # FIX: guard against the edge case where no trades were generated at all
    if trades.empty:
        return pd.DataFrame(
            columns=[
                "trade_id", "entry_date", "entry_spot",
                "exit_date", "exit_spot", "result", "pnl", "holding_days",
            ]
        )

    trades = trades[
        ["trade_id", "entry_date", "entry_spot",
         "exit_date", "exit_spot", "result", "pnl", "holding_days"]
    ]
    trades = trades.sort_values(["entry_date", "trade_id"]).reset_index(drop=True)
    return trades


trades = backtest_short_signal(df, allow_multiple_positions=ALLOW_MULTIPLE_POSITIONS)
trades.head(20)

In [ ]:
def backtest_short_signal(
    data: pd.DataFrame,
    tp_pct: float,
    sl_pct: float,
    holding_period: int,
    allow_multiple_positions: bool = False,
) -> pd.DataFrame:
    """
    Run the short-signal backtest.

    Parameters
    ----------
    data : pd.DataFrame
        Must contain columns: close, low, high, sma_50, sma_200, signal.
    tp_pct : float
        Take-profit threshold as a decimal (e.g. 0.01 = 1%).
        TP triggered when abs(low / entry_close - 1) >= tp_pct.
    sl_pct : float
        Stop-loss threshold as a decimal (e.g. 0.01 = 1%).
        SL triggered when abs(high / entry_close - 1) >= sl_pct.
    holding_period : int
        Maximum number of trading days to hold a position.
        On day holding_period, position is force-closed:
          - TP if entry_close > low  (price fell — short is profitable)
          - SL if entry_close < high (price rose — short is underwater)
    allow_multiple_positions : bool
        If False (default), only one position open at a time.
        If True, a new short is opened on every +1 signal day.

    Returns
    -------
    pd.DataFrame
        Input dataframe with 3 additional columns appended:
          - status       : 'TP', 'SL', or 'OPEN' for the trade opened on that row
          - pnl          : entry_close - exit_level (positive = profitable short)
          - holding_days : number of trading days the position was held
        Rows with no trade entry are NaN in all three columns.
    """
    # Work on a copy; pre-allocate result columns
    out = data.copy()
    out["status"]       = np.nan
    out["pnl"]          = np.nan
    out["holding_days"] = np.nan

    # Map date → integer iloc for fast trading-day counting
    dates = list(out.index)
    date_to_iloc = {d: i for i, d in enumerate(dates)}

    open_positions: List[Dict] = []   # each dict: {entry_iloc, entry_date, entry_close}

    for iloc, (date, row) in enumerate(out.iterrows()):
        close  = row["close"]
        low    = row["low"]
        high   = row["high"]
        signal = row["signal"]

        # ------------------------------------------------------------------
        # 1) Exit logic — check each open position against today's bar
        #    Order: check TP/SL first; if neither fires, check forced close.
        # ------------------------------------------------------------------
        still_open: List[Dict] = []

        for pos in open_positions:
            trading_days_held = iloc - pos["entry_iloc"]  # 0 on entry day itself
            entry_close       = pos["entry_close"]
            entry_row_iloc    = pos["entry_iloc"]

            exit_reason: Optional[str] = None
            exit_level:  Optional[float] = None

            # --- within holding period: standard TP / SL ---
            if trading_days_held < holding_period:
                tp_hit = abs(low  / entry_close - 1) >= tp_pct
                sl_hit = abs(high / entry_close - 1) >= sl_pct

                if tp_hit:
                    exit_reason = "TP"
                    # exit at the TP level itself: entry_close * (1 - tp_pct)
                    exit_level  = entry_close * (1 - tp_pct)
                elif sl_hit:
                    exit_reason = "SL"
                    # exit at the SL level itself: entry_close * (1 + sl_pct)
                    exit_level  = entry_close * (1 + sl_pct)

            # --- forced close on / after holding_period ---
            else:
                if entry_close > low:       # price fell  → short is profitable → TP
                    exit_reason = "TP"
                    exit_level  = low
                else:                       # price rose  → short is losing     → SL
                    exit_reason = "SL"
                    exit_level  = high

            if exit_reason is None:
                still_open.append(pos)
            else:
                pnl = entry_close - exit_level   # short: profit when exit_level < entry
                # Write result back to the entry row
                out.at[pos["entry_date"], "status"]       = exit_reason
                out.at[pos["entry_date"], "pnl"]          = round(pnl, 6)
                out.at[pos["entry_date"], "holding_days"] = trading_days_held

        open_positions = still_open

        # ------------------------------------------------------------------
        # 2) Entry logic
        # ------------------------------------------------------------------
        can_enter = allow_multiple_positions or len(open_positions) == 0

        if signal == 1 and can_enter:
            open_positions.append(
                {
                    "entry_iloc":  iloc,
                    "entry_date":  date,
                    "entry_close": close,
                }
            )

    # ------------------------------------------------------------------
    # 3) Mark any still-open positions at end of data
    # ------------------------------------------------------------------
    for pos in open_positions:
        out.at[pos["entry_date"], "status"]       = "OPEN"
        out.at[pos["entry_date"], "pnl"]          = np.nan
        out.at[pos["entry_date"], "holding_days"] = len(dates) - 1 - pos["entry_iloc"]

    return out


# --- Example call ---
result = backtest_short_signal(
    df,
    tp_pct=0.01,           # 1% take profit
    sl_pct=0.01,           # 1% stop loss
    holding_period=10,     # max 10 trading days
    allow_multiple_positions=False,
)

# Show only rows where a trade was opened
result[result["status"].notna()]

## 5) Trade records

This table gives:

- entry date / exit date
- whether the trade ended in `TP`, `SL`, or is still `OPEN`
- realized PnL per trade (in spot price units)
- holding period in calendar days


In [ ]:
trade_records = trades.copy()
trade_records

## 6) Summary statistics

Includes:

- TP / SL / OPEN counts and rates
- Total, average, and median PnL for closed trades
- Average and median holding period


In [ ]:
def summarize_trades(trades: pd.DataFrame) -> pd.DataFrame:
    # FIX: guard against empty trades DataFrame to avoid ZeroDivisionError
    if trades.empty:
        return pd.DataFrame({"metric": ["total_trades"], "value": [0]})

    total_trades  = len(trades)
    tp_count      = int((trades["result"] == "TP").sum())
    sl_count      = int((trades["result"] == "SL").sum())
    open_count    = int((trades["result"] == "OPEN").sum())

    closed        = trades[trades["result"].isin(["TP", "SL"])].copy()
    closed_total  = len(closed)

    # FIX (improvement): PnL and holding-period stats on closed trades only
    total_pnl     = round(closed["pnl"].sum(), 6)         if closed_total > 0 else np.nan
    avg_pnl       = round(closed["pnl"].mean(), 6)        if closed_total > 0 else np.nan
    median_pnl    = round(closed["pnl"].median(), 6)      if closed_total > 0 else np.nan
    avg_hold      = round(closed["holding_days"].mean(), 1)   if closed_total > 0 else np.nan
    median_hold   = round(closed["holding_days"].median(), 1) if closed_total > 0 else np.nan

    summary = pd.DataFrame(
        {
            "metric": [
                "total_trades",
                "tp_count",
                "sl_count",
                "open_count",
                "tp_rate_over_total",
                "sl_rate_over_total",
                "closed_trades",
                "tp_rate_over_closed",
                "sl_rate_over_closed",
                "total_pnl",
                "avg_pnl_per_trade",
                "median_pnl_per_trade",
                "avg_holding_days",
                "median_holding_days",
            ],
            "value": [
                total_trades,
                tp_count,
                sl_count,
                open_count,
                tp_count / total_trades  if total_trades  > 0 else np.nan,
                sl_count / total_trades  if total_trades  > 0 else np.nan,
                closed_total,
                tp_count / closed_total  if closed_total  > 0 else np.nan,
                sl_count / closed_total  if closed_total  > 0 else np.nan,
                total_pnl,
                avg_pnl,
                median_pnl,
                avg_hold,
                median_hold,
            ],
        }
    )
    return summary


summary = summarize_trades(trades)
summary

## 7) Optional: export trade blotter to CSV

In [ ]:
# Uncomment to export
# trade_records.to_csv("trade_blotter.csv", index=False)
# summary.to_csv("summary_stats.csv", index=False)
# print("Exported trade_blotter.csv and summary_stats.csv")


## 8) Quick sanity check

In [ ]:
display(df.tail(10))
display(trade_records.tail(10))
display(summary)

## Change log

| # | Cell | Issue / Change | Action |
|---|------|----------------|--------|
| 1 | `get_spot_data` | Originally fetched `px_low` mislabelled as `px_last`; later fetched both fields | Now fetches only `px_low` → `low`; `px_last` dropped entirely |
| 2 | `prepare_signals` | SMAs were built on `spot` (PX_LAST) | SMAs now built on `low` (PX_LOW); signal fires when `px_low > SMA50(low)` |
| 3 | `prepare_signals` | Look-ahead bias: same-bar signal used for same-bar entry | Signal `.shift(1)`; row-0 NaN → `.fillna(-1)` |
| 4 | `backtest_short_signal` | Entry price used `spot` (PX_LAST) | Entry price now uses `low` (PX_LOW); `spot` variable removed from loop |
| 5 | `backtest_short_signal` | Exit conditions evaluated against `spot` | Exit conditions evaluated against `low`; exit price recorded as `low` |
| 6 | `backtest_short_signal` | Every `+1` day opened a new overlapping short | `ALLOW_MULTIPLE_POSITIONS` flag; default = one position at a time |
| 7 | `backtest_short_signal` | No PnL or holding-period tracking | `pnl = entry_low - exit_low`; `holding_days` added per closed trade |
| 8 | `backtest_short_signal` | Empty trades caused `KeyError` on column selection | Empty-DataFrame guard added |
| 9 | `summarize_trades` | No PnL / holding-period summary; no empty-trades guard | Stats added; early-return guard added |
